# Grid fill: greedy restart vs backtracking (mini case study)

GridGPT fills a crossword template with real database words such that every crossing letter agrees. This is a **constraint satisfaction problem** (slots are variables, their domains are words of the right length, constraints are equal letters at intersections).

The original filler used **greedy fill with full restarts**: fill every slot in one pass, and on the first dead end throw away the whole grid and start over. This was replaced with a proper **backtracking search** (most-constrained-slot-first ordering, forward checking, and an indexed word list). Both approaches are preserved so we can compare them side by side:

- `src/gridgpt/crossword_generator_legacy.py` - the frozen greedy-restart snapshot
- `src/gridgpt/crossword_generator.py` - the backtracking filler used in production

This notebook runs both on the same tasks and compares success rate and speed. It uses no LLM or embedding calls; "themes" are simulated with a fixed seed word pinned into a slot, which stresses the fill exactly like a real theme entry does.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, logging, random
import pandas as pd

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

logging.getLogger().setLevel(logging.ERROR)

In [ ]:
from src.gridgpt.word_database_manager import WordDatabaseManager
from src.gridgpt.template_manager import load_templates
from scripts.evaluate_generation import run_config, build_generate_fn

word_db = WordDatabaseManager()
templates = {t['id']: t for t in load_templates()['templates']}

SEED = 'APPLE'   # a real 5-letter database word, pinned like a theme entry
RUNS = 8         # generations per (algorithm, template)
len(templates)

## 1. Comparing performance

To better understand the performance difference, we run both algorithms on the same set of tasks and measure their success rate and timing.



### Run both algorithms on the same tasks

The legacy filler is given a bounded attempt budget (20 outer x 50 inner passes) so its failures resolve in seconds instead of thrashing for ~50s at its 100x100 default. The backtracking filler uses its production config.

In [ ]:
rows = []
for algorithm in ['legacy', 'backtracking']:
    random.seed(1)  # same random stream per algorithm for a fair comparison
    generate_fn = build_generate_fn(algorithm, word_db, legacy_outer=20, legacy_inner=50)
    for template_id, template in templates.items():
        stats = run_config(template, SEED, RUNS, generate_fn, word_db)
        rows.append({
            'algorithm': algorithm,
            'template': template_id,
            'success %': round(stats['success_rate'] * 100),
            'mean ms': round(stats['mean_ms'], 1),
            'p95 ms': round(stats['p95_ms'], 1),
        })

comparison = pd.DataFrame(rows)
comparison

In [ ]:
# Aggregate: mean success and time per algorithm
comparison.groupby('algorithm')[['success %', 'mean ms', 'p95 ms']].mean().round(1)

### Review the results

#### What the numbers show

On a representative run (30 generations per config), the two approaches land roughly here:

| | greedy restart (legacy) | backtracking |
|---|---|---|
| success rate | ~67% (even given 1000 attempts) | 100% |
| mean time | hundreds to thousands of ms | ~1-3 ms |
| p95 time | up to ~6 s | under ~6 ms |

That is roughly a **1000x speedup** and a jump from partial to complete reliability. At its real production default (100x100), the legacy filler is far worse still: a single infeasible seed can spin for ~50 seconds before giving up.

#### Why backtracking wins

The greedy filler discards *all* progress the moment one slot has no options, so it re-solves the easy part of the grid over and over. Backtracking instead undoes only the **last** placement and tries the next candidate. Three key components make it both fast and reliable:

- **Most-constrained-variable (MRV) ordering** - always fill the slot with the fewest remaining options, so dead ends surface early and cheaply.
- **Forward checking** - after placing a word, verify every crossing slot still has at least one option; if not, backtrack immediately instead of discovering it deep in the search.
- **Indexed word list** - a `(length, position, letter) -> words` index turns "5-letter words with E in position 2" into a set intersection instead of a scan of thousands of words.

Candidate words are still tried in a frequency-weighted **random** order, so every generation yields a different puzzle. This is what makes a future work package on theme-weighted fill viable: skewing the weights toward on-theme words no longer risks the fill grinding to a halt.

## 2. Comparing fill approaches

To better understand what's happening under the hood, this section steps through the actual code, one operation at a time, so you can see *how* each algorithm works.

In [ ]:
from src.gridgpt.crossword_generator import CrosswordGenerator
from src.gridgpt.crossword_generator_legacy import LegacyCrosswordGenerator

demo_template = templates['5x5_blocked_corners']
entry = 'APPLE' # a real 5-letter database word, used as the theme entry

display(demo_template['grid'])


### The backtracking filler (current)

A theme entry may only go in a slot of matching length. `find_suitable_slots` lists the options and `place_theme_entry` pins the word into one (chosen at random here).

In [ ]:
generator = CrosswordGenerator(word_db)
random.seed(0)

# A theme entry may only go in a slot of its own length.
print('Valid theme entry?', generator.validate_theme_entry(entry))
print(f'{entry} fits slots:', [s['id'] for s in generator.find_suitable_slots(demo_template, entry)])

# place_theme_entry pins it into one of those slots (chosen at random).
seeded = generator.place_theme_entry(demo_template, entry)
theme_slot = list(seeded['filled_slots'])[0]
print(f'Placed {entry} in slot {theme_slot}')
display(seeded['grid'])

In [ ]:
# Every crossing slot is now constrained by the pinned letters 
# Backtracking has no fixed slot order; at each step it expands the MOST constrained slot (MRV).
intersections = generator._build_intersection_map(demo_template)
lengths = {s['id']: s['length'] for s in demo_template['slots']}
assignment = dict(seeded['filled_slots'])
used = set(assignment.values())

# Based on the current assignment, how many candidates remain for each unfilled slot?
candidate_counts = {
    sid: len(generator._candidate_words(lengths[sid], generator._fixed_letters(sid, assignment, intersections), used))
    for sid in lengths if sid not in assignment
}

print('Candidates per unfilled slot (MRV fills the smallest first):')
for sid, n in sorted(candidate_counts.items(), key=lambda kv: kv[1]):
    print(f'  {sid}: {n:>5}   forced letters: {generator._fixed_letters(sid, assignment, intersections)}')

print()
print()
    
# Candidates come from a set-intersection over the (length, position, letter)
# index, tried in a frequency-weighted random order (keeps every puzzle varied).
mrv_slot = min(candidate_counts, key=candidate_counts.get)
forced = generator._fixed_letters(mrv_slot, assignment, intersections)
candidates = generator._candidate_words(lengths[mrv_slot], forced, used)
order = generator._weighted_order(candidates, lambda w: word_db.word_frequencies.get(w, 1))
print(f'\nMost constrained slot: {mrv_slot} (forced {forced}, {len(candidates)} candidates)')
print('First 8 words tried:', order[:8])

print()
print()

# Try each candidate in the MRV slot, one at a time, and see if it leads to a solution.
unfilled = {slot['id'] for slot in seeded['slots'] if slot['id'] not in seeded['filled_slots']}
remaining = unfilled - {mrv_slot}
neighbors = [nid for nid, _, _ in intersections[mrv_slot] if nid in remaining]

for word in order[:8]:
    assignment[mrv_slot] = word
    used.add(word)
    print(f'\nTrying {word} in slot {mrv_slot}')
    
    # Placing a word triggers forward checking: each crossing slot must keep >= 1
    # candidate, else the word is rejected and the next is tried. 
    ok = True
    for neighbor_id in neighbors:
        fixed = generator._fixed_letters(neighbor_id, assignment, intersections)
        if not generator._candidate_words(lengths[neighbor_id], fixed, used):
            print(f'  {neighbor_id} has no candidates with forced letters {fixed}')
            ok = False
            break
    
    if ok:
        # Recursing fills the grid, undoing only the last placement on a dead end.
        print(f'  {word} is OK; {len(neighbors)} neighbors have candidates')
        result = generator._backtrack(assignment, remaining, lengths, intersections, used, lambda w: word_db.word_frequencies.get(w, 1), 100, [0])
        if result is not None:
            print(f'  {word} led to a solution!')
            print(result)
    
    del assignment[mrv_slot]
    used.remove(word)

In [ ]:
# fill() runs the full search
random.seed(0)
solution = generator.fill(demo_template, seed_assignment=dict(seeded['filled_slots']))
print('Completed fill:')
for sid, word in sorted(solution.items()):
    print(f"  {sid}: {word}{'  <- theme entry' if sid == theme_slot else ''}")

### The legacy filler: greedy with restarts

The original filler shared the "fill constrained slots first" intuition but with two crucial differences: it ordered slots **once**, statically, by how many other slots they cross, and on the first dead end it **discarded the entire grid and started over**.

That restart-everything strategy is the source of the gap in the metrics above: a single unlucky word near the end wastes the entire pass, so the legacy filler is both slow and prone to failing within a limited budget. Backtracking keeps all the good work and reconsiders only the last decision, which is why it fills reliably in a couple of milliseconds.

In [ ]:
legacy = LegacyCrosswordGenerator(word_db)
random.seed(0)

# place_theme_entry pins theme entry into one slot chosen at random.
legacy_seeded = legacy.place_theme_entry(demo_template, entry)
theme_slot = list(legacy_seeded['filled_slots'])[0]
print(f'Placed {entry} in slot {theme_slot}')
display(legacy_seeded['grid'])

In [ ]:
# Static order: computed once, by crossing count, and never re-evaluated
crossing_counts = {
    s['id']: len(legacy.get_intersecting_slots(demo_template, s['id']))
    for s in demo_template['slots'] if s['id'] not in legacy_seeded['filled_slots']
}
print('Legacy static order (most crossings first):')
for sid, n in sorted(crossing_counts.items(), key=lambda kv: -kv[1]):
    print(f'  {sid}: {n} crossings')

In [ ]:
# One greedy pass fills each slot in that fixed order. If any slot runs out of
# options, the whole pass returns None and all placed words are thrown away.
one_pass = legacy.fill_grid_with_constraints(legacy_seeded)
print('\nOne greedy pass:', 'filled' if one_pass else 'DEAD END (all progress discarded, restart)')

In [ ]:
# backtracking_fill just repeats the pass until one happens to succeed
filled = legacy.backtracking_fill(legacy_seeded, max_attempts=200)
print('Succeeded after repeated restarts?', filled is not None)